# 07 - Reaction-Diffusion Equation

In this notebook, we solve the reaction-diffusion equation using PINNs:

$$\frac{\partial u}{\partial t} = D \frac{\partial^2 u}{\partial x^2} + R(u)$$

This equation models chemical reactions, pattern formation (Turing patterns), and biological systems.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt
from pinns import Config, MLP, Trainer, set_seed
from pinns.equations.reaction_diffusion import ReactionDiffusion1D
from pinns.evaluation import plot_solution_1d, compute_all_metrics

In [ ]:
set_seed(42)

# Simple reaction: R(u) = u(1-u) (logistic growth)
def reaction_fn(u):
    return u * (1 - u)

config = Config(
    name='reaction_diffusion_1d',
    equation_name='reaction_diffusion_1d',
    equation_params={'D': 0.01, 'reaction_fn': reaction_fn},
    model={'type': 'mlp', 'hidden_layers': [32, 32, 32], 'activation': 'tanh'},
    training={'n_collocation': 1000, 'n_boundary': 100, 'n_initial': 100, 'epochs': 3000, 'lr': 1e-3},
)

eq = ReactionDiffusion1D(D=0.01, reaction_fn=reaction_fn)
print(f"Domain: {eq.domain}")

In [ ]:
model = MLP(
    input_dim=2,  # [x, t]
    output_dim=1,
    hidden_layers=[32, 32, 32],
    activation='tanh'
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
trainer = Trainer(
    model=model,
    equation=eq,
    config=config,
    print_every=500
)

history = trainer.train()

plt.figure(figsize=(10, 4))
plt.plot(history['total_loss'], label='Total Loss')
plt.plot(history['physics_loss'], label='Physics Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.yscale('log')
plt.legend()
plt.title('Reaction-Diffusion Training')
plt.show()

In [ ]:
import numpy as np

# Create space-time grid for visualization
x = torch.linspace(0, 1, 100).reshape(-1, 1)
t = torch.full_like(x, 0.5)

with torch.no_grad():
    u_pred = model(torch.cat([x, t], dim=1))

# Simple analytical approximation for comparison
u_exact = 1 / (1 + torch.exp(-(x - 0.5) / (2 * 0.01)))

plot_solution_1d(
    x=x.numpy(),
    u_pred=u_pred.numpy(),
    u_true=u_exact.numpy(),
    title='Reaction-Diffusion at t=0.5'
)

In [ ]:
metrics = compute_all_metrics(u_pred.numpy(), u_exact.numpy())
print("\nError Metrics:")
for name, value in metrics.items():
    print(f"  {name}: {value:.6e}")